# SONA — Phase A remote batch

Three tasks the local machine cannot do well.

| id | task | why here |
|---|---|---|
| **R2** | ONNX timing: FP32 vs uniform INT8 vs stem-exempt mixed | the local box runs ~20 % background load, so absolute RTF is not quotable there. Kaggle is dedicated **and** Intel Xeon against a local AMD Ryzen, so this doubles as a cross-architecture replication of "INT8 is slower than FP32" |
| **W2** | WER on MS-SNSD (1100 clips) | needs a GPU Whisper pass |
| **W3** | WER on VoiceBank with `tiny.en` | needs a GPU Whisper pass |

**Why W2/W3 exist.** The per-clip WER tests came out underpowered: Whisper `small.en` transcribes
most of these short utterances perfectly under both methods, so only 129/824 pairs differ at all
and per-condition effective n falls to 16–37. W2 adds clips; W3 raises the error floor by using a
weaker recogniser. Both attack the same problem.

---

### Setup

Attach **one** dataset — the folder produced by `scripts/stage_kaggle_dataset.py`. Its manifests
carry paths relative to the dataset root, so nothing needs to be searched for.

Turn the **GPU on** (Settings → Accelerator → GPU T4) for W2/W3. R2 is CPU-only by design.


In [ ]:
%pip install -q onnxruntime==1.20.1 faster-whisper jiwer pystoi

import csv, glob, json, os, subprocess, sys, time
from pathlib import Path

IN_KAGGLE = Path('/kaggle/input').exists()
OUT = Path('/kaggle/working/phase_a' if IN_KAGGLE else './phase_a')
OUT.mkdir(parents=True, exist_ok=True)

# Locate the staged dataset by a unique marker file, at any nesting depth.
# Kaggle may place it several levels below /kaggle/input (e.g. datasets/<user>/<name>).
def find_data_root():
    base = Path('/kaggle/input') if IN_KAGGLE else Path('kaggle_upload')
    hits = sorted(base.glob('**/onnx/model_mixed.onnx'))
    return hits[0].parents[1] if hits else None

DATA = find_data_root()
assert DATA, 'attach the staged dataset (must contain repo/ and onnx/)'
REPO = str(DATA / 'repo')
CKPT = str(DATA / 'ckpt' / 'last.pt')
print('DATA =', DATA)
print('REPO =', REPO)
print('CKPT exists =', Path(CKPT).exists())

In [ ]:
# Manifests hold dataset-relative paths; make them absolute for this session.
def absolutize(rel_csv, cols, out_name):
    src = DATA / rel_csv
    rows = list(csv.DictReader(src.open(newline='', encoding='utf-8')))
    missing = 0
    for r in rows:
        for c in cols:
            p = DATA / r[c]
            if not p.exists():
                missing += 1
            r[c] = str(p)
    dst = OUT / out_name
    with dst.open('w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print(f'{rel_csv}: {len(rows)} rows, {missing} missing -> {dst}')
    return str(dst)

VBD_MANIFEST = absolutize('vbd/test_manifest.csv', ['noisy', 'clean'], 'vbd_test.csv')
MS_MANIFEST  = absolutize('ms_snsd/test.csv',      ['clean', 'noisy'], 'ms_snsd_test.csv')

## Machine identity

Recorded so the timing numbers can be attributed. Any absolute RTF the paper quotes must carry CPU
and load — that was the fix applied after the first RTF pass proved load-inflated.


In [ ]:
import platform
info = {'platform': platform.platform(), 'python': sys.version.split()[0]}
try:
    info['cpu_model'] = [l.split(':',1)[1].strip() for l in open('/proc/cpuinfo')
                         if l.startswith('model name')][0]
    info['cpu_count'] = sum(1 for l in open('/proc/cpuinfo') if l.startswith('processor'))
    info['loadavg_1min'] = float(open('/proc/loadavg').read().split()[0])
except Exception as e:
    info['cpuinfo_error'] = str(e)
print(json.dumps(info, indent=2))
(OUT / 'machine.json').write_text(json.dumps(info, indent=2))

## R2 — timing: FP32 vs uniform INT8 vs stem-exempt mixed  *(~10 min, CPU)*

The claim under test: mixed carries **29 QDQ nodes against uniform INT8's 172**. If QDQ graph
overhead is what makes INT8 slower than FP32 here, mixed should sit between them — and if it beats
uniform, the stem exemption is smaller *and* faster, turning a negative result into a positive one.


In [ ]:
cmd = [sys.executable, str(Path(REPO) / 'scripts' / 'bench_onnx_edge.py'),
       '--checkpoint', CKPT,
       '--onnx-dir', str(DATA / 'onnx'),
       '--variants', 'fp32,int8,mixed',
       '--manifest', VBD_MANIFEST,
       '--limit', '100',
       '--threads', '1,2,4',
       '--out', str(OUT / 'r2_timing')]
print(' '.join(cmd), '\n')
subprocess.run(cmd, check=True, cwd=REPO)

In [ ]:
import pandas as pd
df = pd.read_csv(OUT / 'r2_timing' / 'summary.csv')
display(df.sort_values(['backend', 'threads']))

fp32 = df[df.backend == 'fp32'].set_index('threads')['mean_rtf']
print('\nslowdown vs FP32 at matched threads (>1 = slower):')
for b in ['int8', 'mixed']:
    sub = df[df.backend == b].set_index('threads')['mean_rtf']
    for t in sub.index:
        if t in fp32.index:
            print(f'  {b:6s} threads={t}  {sub[t]/fp32[t]:.3f}x')

## W2 — WER on MS-SNSD (1100 clips)  *(~35–50 min, GPU)*

Roughly doubles the ASR sample and adds noise categories VoiceBank does not cover.


In [ ]:
cmd = [sys.executable, str(Path(REPO) / 'scripts' / 'eval_wer.py'),
       '--manifest', MS_MANIFEST, '--limit', '0',
       '--whisper-model', 'small.en',
       '--gtcrn-checkpoint', CKPT,
       '--out', str(OUT / 'w2_wer_ms_snsd')]
print(' '.join(cmd), '\n')
subprocess.run(cmd, check=True, cwd=REPO)
print((OUT / 'w2_wer_ms_snsd' / 'summary_wer.md').read_text())

## W3 — WER on VoiceBank with a weaker ASR model  *(~20–30 min, GPU)*

`small.en` is near-ceiling on this corpus, which is *why* the per-condition tests were
underpowered. `tiny.en` raises the error floor so more clips carry signal.

Reporting both is the honest framing: whether enhancement helps ASR is genuinely
ASR-model-dependent, and that dependence is itself on-thesis for a paper about metric choice
changing conclusions.


In [ ]:
cmd = [sys.executable, str(Path(REPO) / 'scripts' / 'eval_wer.py'),
       '--manifest', VBD_MANIFEST, '--limit', '0',
       '--whisper-model', 'tiny.en',
       '--gtcrn-checkpoint', CKPT,
       '--out', str(OUT / 'w3_wer_tiny')]
print(' '.join(cmd), '\n')
subprocess.run(cmd, check=True, cwd=REPO)
print((OUT / 'w3_wer_tiny' / 'summary_wer.md').read_text())

In [ ]:
# effective-sample check: the whole point of W3
import pandas as pd
d = pd.read_csv(OUT / 'w3_wer_tiny' / 'per_sample_wer.csv')
piv = d.pivot_table(index='uid', columns='method', values='wer')
if {'neural_gtcrn', 'passthrough'} <= set(piv.columns):
    diff = (piv['neural_gtcrn'] - piv['passthrough']).dropna()
    print(f'clips {len(diff)}   differing {int((diff != 0).sum())} '
          f'({100*(diff != 0).mean():.1f} %)')
    print('small.en reference was 129/824 = 15.7 %')

## Bundle the outputs

Download `phase_a_results.zip` from the Kaggle output panel and hand it back — that is everything
needed to finish the paper bundle.


In [ ]:
import shutil
base = '/kaggle/working/phase_a_results' if IN_KAGGLE else './phase_a_results'
shutil.make_archive(base, 'zip', OUT)
print('wrote', base + '.zip')